Recall that the coboundary operator is

$$\partial: C^k(\mathfrak{m},\mathfrak{g})\to C^{k+1}(\mathfrak{m},\mathfrak{g})$$

and is defined by

$$\partial\phi(\alpha_0,\ldots,\alpha_{k}) = \sum_{i=0}^{k}(-1)^i\big[\alpha_i,\phi(\alpha_0,\ldots,\hat\alpha_i,\ldots, \alpha_{k})\big]$$
$$+ \sum_{i<j}(-1)^{i+j}\phi([\alpha_i,\alpha_j],\alpha_0,\ldots, \hat \alpha_i,\ldots,\hat\alpha_j,\ldots,\alpha_{k})$$


We consider the inner product on the Tanaka symbol with orthonormal basis $(Y,H,E,X,\varepsilon_1,\ldots,\varepsilon_{2m},\eta)$ and lengths

$$|Y|^2=|X|^2=1,|H|^2=|E|^2=2,|\varepsilon_i|^2=\frac{(i-1)!}{(2m-i)!}, |\eta|^2=1$$

along with the innerproduct induced on tensor spaces. In particular, 

$$|A^*\wedge B^*\otimes C|^2 = \frac{|C|^2}{|A|^2|B|^2}$$

In [74]:
%run T_symb.ipynb
%run helpers.ipynb
import copy
from sympy import *
from itertools import combinations

In [75]:
def key_from_neg(c_key):
    '''arg: c_key a tuple of strings like ('H','X','e1')
       returns: True if c_key is from the complex C(g_-,g), False otherwise.'''
    for i in range(len(c_key)-1):
        A=c_key[i]
        if A in ('H','E','Y'): return False
    return True

In [124]:
class cochain_complex:
    def __init__(self,T_symb_obj,pickled_ad_degs=[]):
        self.pickled_ad_degs=[]
        self.alg=T_symb_obj
        self.heis_dim=T_symb_obj.heis_dim
        T_symb_obj.cochain_complex=self
        self.ext_alg=ext_alg(T_symb_obj,pickled_ad_degs)
        T_symb_obj.ext_alg=self.ext_alg
        
        self.basis={}
        self.basis_strs={}
        
        self.iprod_matrices={}
                
        self.ker_iprod_matrices={}
        self.im_iprod_matrices={}
        
        self.coboundary_dicts={}
                
        self.coboundary_matrices={}
        self.coboundary_im={}
        self.coboundary_im_vecs={}
        self.coboundary_preim={}
        self.coboundary_preim_vecs={}
        self.coboundary_ker={}
        self.coboundary_ker_vecs={}
        
        self.coker={}
        self.coker_vecs={}
        
        self.harm_basis={}

    
    def cochain(self,coeff_dict={},wght='UNKNOWN',deg='UNKNOWN'):
        return cochain(coeff_dict,self,wght,deg)
    
    def sort_tuple(self,cochain_tuple):
        '''cochain_tuple: a tuple of str_reps of T_symb_basis_elt objs, representing a cochain
           returns: a rearrangement of cochain_tuple, descending in degree,
                    but leaving the final element of cochain_tuple invariant'''
        ext_list=list(cochain_tuple[0:len(cochain_tuple)-1])
        ext_result=self.alg.sort_basis_tuple(ext_list)
        return((tuple(list(ext_result[0])+[cochain_tuple[len(cochain_tuple)-1]]),ext_result[1]))
    
    def deg(self,c):
        '''c: a cochain object from self
           returns: deg(c) if c has homogeneous deg, 
             'Nil' if c is the zero cochain,'UNKNOWN' otherwise'''
        if len(c.coeff_dict.keys())==0:
            return 'Nil' # The zero cochain
        deg=len(list(c.coeff_dict.keys())[0])-1
        for A in c.coeff_dict:
            if len(A)-1!=deg:
                return 'UNKNOWN'
        return deg
    
    def tuple_wght(self,rep):
        '''rep: a tuple of strings representing a basic cochain from self
           returns: the wght of the corresponding cochain'''
        result=0
        for A in rep[0:len(rep)-1]:
            result-=self.alg.basis[self.alg.basis_strs.index(A)].wght
        result+=self.alg.basis[self.alg.basis_strs.index(rep[len(rep)-1])].wght
        return result
    
    def wght(self,c):
        '''c: a cochain object
           returns: wght(c) if c has homogeneous wght, 'UNKNOWN' otherwise'''
        if len(c.coeff_dict.keys())==0:
            return 'Nil' # The zero cochain
        wght=self.tuple_wght(list(c.coeff_dict.keys())[0])
        for A in c.coeff_dict:
            if self.tuple_wght(A)!=wght:
                return 'UNKNOWN'
        return wght 
    
    def init_basis(self,deg):
        if deg in self.basis: return None
        deg_subsets=list(combinations(self.alg.neg_basis_strs,deg))
        
        cochain_subsets=[A+(B,) for A in deg_subsets for B in self.alg.basis_strs]
        self.basis[deg]={}
        self.basis_strs[deg]={}
        for A in cochain_subsets:
            wght=self.tuple_wght(A)
            if wght in self.basis[deg]:
                self.basis_strs[deg][wght].append(A)
                self.basis[deg][wght].append(self.cochain({A:1}))
            else:
                self.basis_strs[deg][wght]=[A]
                self.basis[deg][wght]=[self.cochain({A:1})]     
    
    def set_coboundary_dict(self,deg):
        # I could certainly speed this up; this is just the most 
        # mathematical/safe way to write this block of code
        self.coboundary_dicts[deg]={}
        # To do: check degree zero
        
        # initialize the bases
        self.init_basis(deg)
        self.ext_alg.init_basis(deg+1)
        
        for wght in self.basis[deg]:
            self.coboundary_dicts[deg][wght]={}
            for c in self.basis[deg][wght]:
                c_str=self.basis_strs[deg][wght][self.basis[deg][wght].index(c)]
                dc=self.cochain({})
                
                for e in self.ext_alg.basis[deg+1]:
                    # compute dc(e)
                    dce=self.alg.elt()
                    e_str=self.ext_alg.basis_strs[deg+1][self.ext_alg.basis[deg+1].index(e)]
                    e.deg=self.ext_alg.deg(e)
                    for i in range(e.deg):
                        # (-1)**i*[ai,c(a0,...\hat ai,...ak)]
                        e_no_i=self.ext_alg.elt({tuple(e_str[0:i]+e_str[i+1:len(e_str)]):1})
                        ei_vec=[0]*len(self.alg.basis)
                        ei_vec[self.alg.basis_strs.index(e_str[i])]=1
                        ei=self.alg.elt(ei_vec)
                        im=c.apply_cochain_map(e_no_i)
                        dce+=(-1)**i*self.alg.ad(ei,im)
                    if e.deg>1:
                        for i in range(e.deg):
                            for j in range(i+1,e.deg):
                                # (-1)**(i+j)*c([ai,aj],a0,...,\hat ai,...\hat aj,...ak)
                                e_noij=self.ext_alg.elt({e_str[0:i]+e_str[i+1:j]+e_str[j+1:e.deg]:1})
                                ei_vec=[0]*len(self.alg.basis)
                                ei_vec[self.alg.basis_strs.index(e_str[i])]=1
                                ej_vec=[0]*len(self.alg.basis)
                                ej_vec[self.alg.basis_strs.index(e_str[j])]=1
                                ei_ej=self.alg.ad(self.alg.elt(ei_vec),self.alg.elt(ej_vec))

                                ei_ej_dual=self.ext_alg.elt({(self.alg.basis_strs[k],):ei_ej.vec_rep[k]
                                                            for k in range(len(self.alg.basis))})
                                if e_noij==0:
                                    new_wedge=ei_ej
                                else:
                                    new_wedge=ei_ej_dual.wedge(e_noij)
                                dce+=(-1)**(i+j)*c.apply_cochain_map(new_wedge)
                    dc+=e.wedge(dce)
                self.coboundary_dicts[deg][wght][c_str]=dc
            
    def coboundary(self, c):
        '''c: a cochain with self as parent
           returns: the coboundary map applied to c'''
        if type(c)!=cochain: raise ValueError('coboundary must only be applied to cochains')
        if c.parent!=self: raise invalid_parent_exception
        return c.coboundary()
    
    def set_iprod_matrices(self,deg):
        '''sets the attribute iprod_matrices[deg] for the cochain_complex;
           iprod_matrices is organized by deg then wght'''
        self.init_basis(deg)
        self.iprod_matrices[deg]={}
        for wght in self.basis[deg]:
            self.iprod_matrices[deg][wght]=iprod_mat(self.basis[deg][wght])
        return
    
    def set_ker_iprod_matrices(self,deg):
        '''sets the attribute ker_iprod_matrices[deg] for the cochain_complex;
           ker_iprod_matrices is organized by deg then wght'''
        if deg not in self.coboundary_ker:
            self.set_coboundary_ker(deg)
                    
        self.ker_iprod_matrices[deg]={}
        for wght in self.coboundary_ker[deg]:
            self.ker_iprod_matrices[deg][wght]=iprod_mat(self.coboundary_ker[deg][wght])
        return
    
    def set_im_iprod_matrices(self,deg):
        '''sets the attribute ker_iprod_matrices[deg] for the cochain_complex;
           ker_iprod_matrices is organized by deg then wght'''

        if deg not in self.coboundary_im:
            self.set_coboundary_im(deg)
        self.im_iprod_matrices[deg]={}
        for wght in self.coboundary_im[deg]:
            self.im_iprod_matrices[deg][wght]=iprod_mat(self.coboundary_im[deg][wght])
        return
    
    def set_coboundary_matrices(self,deg):
        '''sets the attribute coboundary_matrices[deg] for the cochain_complex'''
        # Set necessary bases
        if deg not in self.coboundary_matrices:
            self.coboundary_matrices[deg]={}
        self.init_basis(deg)
        self.init_basis(deg+1)
                        
        for wght in self.basis[deg]:
            dom_b=self.basis[deg][wght]
            if wght in self.basis_strs[deg+1]:
                cod_b=self.basis_strs[deg+1][wght]
                result=zeros(len(cod_b),0)
                for i in range(len(self.basis[deg][wght])):
                    c=self.basis[deg][wght][i]
                    im=self.coboundary(c)
                    v=coordinatize_cochain_in_basis(im,cod_b)
                    result=result.col_insert(result.shape[1],Matrix(v))
            else: result=zeros(0,len(dom_b))
            self.coboundary_matrices[deg][wght]=result
            
    def neg_proj(self,c):
        '''projects the cochain c onto the complex C(g_-,g) using ortho proj from
           the inner product where the chosen basis is orthonormal'''
        new_dict=copy.copy(c.coeff_dict)
        for key in c.coeff_dict:
            if not key_from_neg(key):
                new_dict.pop(key)
        return c.parent.cochain(new_dict) 
        
    
    def set_coboundary_im(self,deg):
        '''sets the attributes coboundary_im[deg], coboundary_im_vecs[deg],
           coboundary_preim[deg], and coboundary_preim_vecs[deg] of self,
           which are a basis for the im(d) of degree deg and cochains mapping to it'''
        if deg in self.coboundary_im: return
        
        if deg-1 not in self.basis: self.init_basis(deg-1)
        if deg-1 not in self.coboundary_matrices: self.set_coboundary_matrices(deg-1)
        self.coboundary_im[deg]={}
        self.coboundary_im_vecs[deg]={}
        self.coboundary_preim[deg]={}
        self.coboundary_preim_vecs[deg]={}
        for wght in self.basis[deg]:
            self.coboundary_im[deg][wght]=[]
            self.coboundary_preim[deg][wght]=[]
            if wght in self.basis[deg-1]:
                T=col_sp_and_preim(self.coboundary_matrices[deg-1][wght])
                col_sp=T[1]
                preim=T[0]
                self.coboundary_preim_vecs[deg][wght]=preim
                self.coboundary_im_vecs[deg][wght]=col_sp
                for v in col_sp:
                    c=sum([v[i]*self.basis[deg][wght][i] for i in range(len(v))])
                    self.coboundary_im[deg][wght].append(c)
                for v in preim:
                    c=sum([v[i]*self.basis[deg-1][wght][i] for i in range(len(v))])
                    self.coboundary_preim[deg][wght].append(c)
        
                
    
    def set_coboundary_ker(self,deg):
        '''sets the attributes coboundary_ker[deg] and coboundary_ker_vecs[deg] of self,
           which is a basis for the ker(d) of degree deg'''
        if deg in self.coboundary_ker: return                
        self.coboundary_ker[deg]={}
        self.coboundary_ker_vecs[deg]={}
        self.set_coboundary_matrices(deg)
        for wght in self.basis[deg]:
            self.coboundary_ker[deg][wght]=[]
            null_sp=self.coboundary_matrices[deg][wght].nullspace()
            self.coboundary_ker_vecs[deg][wght]=null_sp
            for v in null_sp:
                c=sum([v[i]*self.basis[deg][wght][i] for i in range(v.shape[0])])
                self.coboundary_ker[deg][wght].append(c) 
        
    def ker_proj(self,c):
        '''c: a cochain object from self
           returns: The orthogonal projection (w.r.t iprod) of c onto ker(d)'''
        result=self.cochain({})
        
        # enumerate all the (deg,wght) pairs involved in c
        dw_set=set()
        for A in c.coeff_dict:
            dw_set.add((self.deg(self.cochain({A:1})),self.tuple_wght(A)))
            
        for dw in dw_set:
            # Make sure ker_iprod_matrices[deg] is set
            if not dw[0] in self.ker_iprod_matrices:
                self.set_ker_iprod_matrices(dw[0])
            v=Matrix([[c.iprod(b) for b in self.coboundary_ker[dw[0]][dw[1]]]])
            a=v*(self.ker_iprod_matrices[dw[0]][dw[1]].inv())
            for i in range(a.shape[1]):
                result+=a[0,i]*self.coboundary_ker[dw[0]][dw[1]][i]
        return result
    
    
    def coker_proj(self,c):
        '''c: a cochain object from self
           returns: The orthogonal projection (w.r.t iprod) of c onto ker(d*)'''
        result=self.cochain(copy.copy(c.coeff_dict))
        
        # enumerate all the (deg,wght) pairs involved in c
        dw_set=set()
        for A in c.coeff_dict:
            dw_set.add((self.deg(self.cochain({A:1})),self.tuple_wght(A)))
            
        for dw in dw_set:
            # Make sure im_iprod_matrices[deg] is set
            if not dw[0] in self.im_iprod_matrices:
                self.set_im_iprod_matrices(dw[0])
            v=Matrix([[c.iprod(b) for b in self.coboundary_im[dw[0]][dw[1]]]])
            a=v*(self.im_iprod_matrices[dw[0]][dw[1]].inv())
            for i in range(a.shape[1]):
                result-=a[0,i]*self.coboundary_im[dw[0]][dw[1]][i]
        return result
    
    def set_coker(self,deg):
        '''sets the attributes coker[deg] and coker_vecs[deg] of self,
           which are a basis for the im(d) of degree deg'''
        if deg in self.coker: return
        if deg not in self.basis: self.init_basis(deg)
                            
        self.coker[deg]={}
        self.coker_vecs[deg]={}
        
        ## Project all elements of self.basis[deg] onto the coker; convert to vectors
        for wght in self.basis[deg]:
            self.coker[deg][wght]=[]
            vecs=[]
            for c in self.basis[deg][wght]:
                vecs.append(coordinatize_cochain_in_basis(self.coker_proj(c),self.basis_strs[deg][wght]))                
        ## Find a basis for their span and convert back to cochains
            col_sp=transpose(Matrix(vecs)).columnspace()
            for v in col_sp:
                c=sum([v[i]*self.basis[deg][wght][i] for i in range(v.shape[0])])
                self.coker[deg][wght].append(c)
    
    
    def harm_proj(self,c):
        '''c: a cochain object from self
           returns: The orthogonal projection (w.r.t iprod) of c onto the harmonic subspace of C(g_-,g)'''
        return self.coker_proj(self.ker_proj(c))
        
    def set_harm_basis(self,deg):
        '''sets the attribute harm_basis of self, which is a basis
           for the harmonic forms organized by deg then wght'''
        
        self.harm_basis[deg]={}
        self.set_coboundary_ker(deg)
        for wght in self.basis[deg]:
            self.harm_basis[deg][wght]=find_cochain_basis([self.coker_proj(A) for A in self.coboundary_ker[deg][wght]])
        return

In [125]:
class ext_alg:
    def __init__(self,T_symb_obj,pickled_ad_degs=[]):
        self.alg=T_symb_obj
        self.cochain_complex=self.alg.cochain_complex
        self.heis_dim=T_symb_obj.heis_dim
        T_symb_obj.ext_alg=self
        self.basis={}
        self.basis_strs={}
        self.neg_basis={}
        self.neg_basis_strs={}
    
    def elt(self,coeff_dict={},wght='UNKNOWN',deg='UNKNOWN'):
        return ext_elt(coeff_dict,self,wght,deg)
    
    def deg(self,ext_elt):
        '''ext_elt: an ext_T_symb_elt object
           returns: deg(ext_elt) if ext_elt has homogeneneous deg,
             'Nil' if ext_elt is the zero wedge, 'UNKNOWN' otherwise'''
        if len(ext_elt.coeff_dict.keys())==0:
            return 'Nil'
        deg=len(list(ext_elt.coeff_dict.keys())[0])
        for A in ext_elt.coeff_dict:
            if len(A)!=deg:
                return 'UNKNOWN'
        return deg
    
    def wedge_tuples(self,tuple1,tuple2):
        '''tuple1,tuple2: tuples of T_symb_basis_elt objects
           returns: (wedge,sgn), where wedge is a tuple representing tuple1 wedge tuple2
                   and sgn is -1 or 1'''
        #check for repeats
        if len(set(tuple1).union(set(tuple2)))!=len(tuple1)+len(tuple2):
            return 'Nil'
        return self.alg.sort_basis_tuple(tuple1+tuple2)
    
    def tuple_wght(self,rep):
        '''rep: a tuple representing an ext_T_symb_elt
           returns: the wght of the corresponding exterior element'''
        result=0
        for A in rep:
            result-=self.alg.basis[self.alg.basis_strs.index(A)].wght
        return result
    
    def wght(self,ext_elt):
        '''ext_elt: an ext_T_symb_elt object
           returns: wght(ext_elt) if ext_elt has homogeneous wght, 'UNKNOWN' otherwise'''
        if len(ext_elt.coeff_dict.keys())==0:
            return 'Nil' # The zero cochain
        wght=self.tuple_wght(list(c.coeff_dict.keys())[0])
        for A in c.coeff_dict:
            if self.tuple_wght(A)!=wght:
                return 'UNKNOWN'
        return wght 
    
    def init_basis(self,deg):
        '''initializes attributes ext_alg.basis[deg], ext_alg.neg_basis[deg] and sets the
        dicts A.ext_ad_dicts[deg] for each A in the algebra's basis '''
        if deg in self.basis: return None
        deg_subsets=list(combinations(self.alg.neg_basis_strs,deg))
        self.basis_strs[deg]=[tuple(A) for A in deg_subsets]
        self.basis[deg]=[self.elt({A:1}) for A in self.basis_strs[deg]]
        

In [126]:
class cochain:
    def __init__(self,coeff_dict,parent,wght='UNKNOWN',deg='UNKNOWN'):
        '''coeff_dict:  tuple of T_symb_basis_elts objs as keys, coeffs as values
           wght (optional): homog. wght, if known'''
        self.parent=parent
        self.deg=deg
        self.wght=wght
        self.heis_dim=parent.heis_dim 
        temp=remove_zeros({parent.sort_tuple(A)[0]:
                                      parent.sort_tuple(A)[1]*coeff_dict[A] for A in coeff_dict})
        self.coeff_dict=remove_antisymm_zeros_cochains(temp)
                
    def __eq__(self,other):
        if other==0:
            return self.coeff_dict=={}
        return self.coeff_dict==other.coeff_dict
        
    def __add__(self,other):
        # zero cochain case
        if other==0 or other.deg=='Nil': return self.parent.cochain(copy.copy(self.coeff_dict))
        if self==0 or self.deg=='Nil': return self.parent.cochain(copy.copy(other.coeff_dict))
        
        result=self.parent.cochain(copy.copy(self.coeff_dict))
        
        ## If one wght is UNKNOWN
        if self.wght==other.wght or other.wght=='UNKNOWN':
            result.wght=self.wght
        elif self.wght=='UNKNOWN':
            result.wght=other.wght
        else:
            result.wght='UNKNOWN'       
        
        result.coeff_dict=merge_coeff_dicts(self.coeff_dict,other.coeff_dict)
            
        return result
    
    def __radd__(self,other):
        return self+other
    
    def __neg__(self):
        return self.parent.cochain({A:-self.coeff_dict[A] for A in self.coeff_dict},
                       wght=self.wght,deg=self.deg)
        
    def __sub__(self,other):
        return self+(-other)
    
    def __mul__(self,k):
        kself=self.parent.cochain(copy.copy(self.coeff_dict))
        kself.coeff_dict={A:k*self.coeff_dict[A] for A in kself.coeff_dict}
        return kself
    
    def __rmul__(self,k):
        return self*k
    
    def __str__(self):
        return str_from_coeff_dict(self.coeff_dict)
    
    def __repr__(self):
        return str_from_coeff_dict(self.coeff_dict)
    
    def __gt__(self,other):
        if self.parent!=other.parent: raise invalid_parent_exception
        return str(self)>str(other)
    
    def __ge__(self,other):
        if self.parent!=other.parent: raise invalid_parent_exception
        return str(self)>=str(other)
    
    def __lt__(self,other):
        if self.parent!=other.parent: raise invalid_parent_exception
        return str(self)<str(other)
    
    def __le__(self,other):
        if self.parent!=other.parent: raise invalid_parent_exception
        return str(self)<=str(other)
    
    # I think these are unecessary now?
    def __getstate__(self):
        return self.__dict__
    
    def __setstate__(self,d):
        self.__dict__=d
        
    def apply_cochain_map(self,e):
        '''e: an ext_T_symb_elt object
           returns: T_symb_elt object representing self(ext_elt) or None if c.deg!=ext_elt.deg'''
        if self.coeff_dict=={} or e.coeff_dict=={}:
            return self.parent.alg.elt()
        result=self.parent.alg.elt()
        for c in self.coeff_dict:
            ext_basis_elt=c[0:len(c)-1]
            if ext_basis_elt in e.coeff_dict:
                coeff=e.coeff_dict[ext_basis_elt]*self.coeff_dict[c]
                v=[0]*len(self.parent.alg.basis)
                v[self.parent.alg.basis_strs.index(c[len(c)-1])]=coeff
                result+=self.parent.alg.elt(v)
        return result
    
    def coboundary(self):
        P=self.parent
        result=P.cochain({})
        for k in self.coeff_dict:
            deg=P.deg(P.cochain({k:1}))
            wght=P.tuple_wght(k)
            try: result+=self.coeff_dict[k]*P.coboundary_dicts[deg][wght][k]
            except:
                P.set_coboundary_dict(deg)
                result+=self.coeff_dict[k]*P.coboundary_dicts[deg][wght][k]
        return result
    
    def iprod(self,c):
        if not hasattr(c,'parent'): raise ValueError('arg of iprod must have parent')
        if self.parent!=c.parent: raise invalid_parent_exception('args of iprod must have the same parent')
        P=self.parent
        result=0
        for A in self.coeff_dict:
            deg=P.deg(P.cochain({A:1}))
            if A in c.coeff_dict:
                mag_A=1
                for b in A[0:len(A)-1]:
                    mag_A=mag_A*Rational(1,P.alg.iprod_list[P.alg.basis_strs.index(b)])
                mag_A=mag_A*P.alg.iprod_list[P.alg.basis_strs.index(A[len(A)-1])]
                result+=self.coeff_dict[A]*c.coeff_dict[A]*mag_A
        return result

In [127]:
class ext_elt():
    
    def __init__(self,coeff_dict,parent,wght='UNKNOWN',deg='UNKNOWN'):
        if coeff_dict=={}: ## The zero wedge has wght and deg 'Nil'
            self.deg='Nil'
            self.wght='Nil'
        else:
            self.deg=deg   
            self.wght=wght
        self.parent=parent
        self.heis_dim=parent.heis_dim
        self.coeff_dict=remove_antisymm_zeros(remove_zeros({parent.alg.sort_basis_tuple(A)[0]:
                                      parent.alg.sort_basis_tuple(A)[1]*coeff_dict[A] for A in coeff_dict}))
    
    def __str__(self):
        return str_from_coeff_dict(self.coeff_dict)
    
    def __repr__(self):
        return str_from_coeff_dict(self.coeff_dict)
    
    def __eq__(self,other):
        if other==0:
            return self.coeff_dict=={}
        return self.coeff_dict==other.coeff_dict
    
    def __neg__(self):
        return self.parent.elt({A:-self.coeff_dict[A] for A in self.coeff_dict}
                               ,wght=self.wght,deg=self.deg)
    
    def __add__(self,other):
        # To Do: add a check for if parents match here (and in similar places)
        if self==0: return self.parent.elt(copy.copy(other.coeff_dict))
        if other==0: return self.parent.elt(copy.copy(self.coeff_dict))
        new_wght='UNKNOWN'
        if self.wght!='UNKNOWN' and other.wght!='UNKNOWN':
            new_wght=self.wght+other.wght
        new_deg='UNKNOWN'
        if self.deg!='UNKNOWN' and other.deg!='UNKNOWN':
            new_deg=self.deg+other.deg
        return(self.parent.elt(merge_coeff_dicts(self.coeff_dict, other.coeff_dict),
                               wght=new_wght,deg=new_deg))
    
    def __radd__(self,other):
        return self+other
    
    def __sub__(self,other):
        return self+(-other)
    
    def __mul__(self,k):
        return(self.parent.elt({A:k*self.coeff_dict[A] for A in self.coeff_dict},
                               wght=self.wght,deg=self.deg))
    
    def __rmul__(self,other):
        return self*other
    
    def wedge(self,other):
        '''other: an ext_T_symb_elt or cochain object
           returns: the wedge product of self and other as an ext_T_symb_elt
           Note: Functionality only for wedges of deg <4, since vec_rep is used'''
        
        if type(other)==ext_elt:
            result=self.parent.elt({})
            for A in self.coeff_dict:
                for B in other.coeff_dict:
                    new_wedge=A+B
                    if new_wedge!='Nil':
                        result+=self.parent.elt({new_wedge:self.coeff_dict[A]*other.coeff_dict[B]})
            return result
        
        if type(other)==cochain:
            result=self.parent.cochain_complex.cochain({})
            for A in self.coeff_dict:
                for B in other.coeff_dict:
                    new_wedge=A+B[0:len(B)-1]
                    new_cochain=new_wedge+B[len(B)-1:len(B)]
                    result+=self.parent.cochain_complex.cochain(
                        {new_cochain:self.coeff_dict[A]*other.coeff_dict[B]})
            return result
        
        if type(other)==T_symb_elt or type(other)==T_symb_basis_elt:
            result=self.parent.cochain_complex.cochain({})
            for e in self.coeff_dict:
                for i in range(len(self.parent.alg.basis)):
                    A=self.parent.alg.basis_strs[i]
                    coeff=self.coeff_dict[e]*other.vec_rep[i]
                    result+=self.parent.cochain_complex.cochain({e+(A,):coeff})
            return result
        
    def iprod(self,e):
        if not hasattr(e,'parent'): raise ValueError('arg of iprod must have parent')
        if self.parent!=e.parent: raise invalid_parent_exception('args of iprod must have the same parent')
        P=self.parent
        result=0
        for A in self.coeff_dict:
            deg=P.deg(P.elt({A:1}))
            if A in e.coeff_dict:
                mag_A=1
                for b in A:
                    mag_A=mag_A*Rational(1,mag_A*P.alg.iprod_list[P.alg.basis_strs.index(b)])
                result+=self.coeff_dict[A]*e.coeff_dict[A]*mag_A
        return result
        
#     # I think these are unecessary now?
#     def __getstate__(self):
#         return self.__dict__
    
#     def __setstate__(self,d):
#         self.__dict__=d
        

In [151]:
T=T_symb(9)
C=T.cochain_complex

In [81]:
C.set_harm_basis(2)

In [82]:
for A in C.harm_basis[2]:
    print(A,'-->',C.harm_basis[2][A])

3 --> [(e2,e3,e2) + 4/3*(e3,e4,e4) - (e1,e4,e2) + -1/3*(e2,e5,e4) + 4/3*(e3,e5,e5) + 3*(e4,e5,e6) + -5/3*(e3,e6,e6) + (e2,e4,e3) + -5/3*(e2,e6,e5) + -5/3*(e1,e6,e4) + -2*(e1,e5,e3)]
2 --> []
1 --> []
0 --> [(e1,e2,e3) + 5/3*(e2,e4,e6) + 5/3*(e2,e3,e5) + -7/3*(e1,e5,e6) + (e1,e3,e4) + -2/3*(e1,e4,e5)]
-1 --> [(e1,e4,e6) + -2/3*(e2,e3,e6) + 1/3*(e1,e2,e4) + 1/3*(e1,e3,e5)]
-2 --> [(e1,e2,e5) + (e1,e3,e6)]
-3 --> [(e1,e2,e6)]
-4 --> []
-5 --> []
4 --> [(X,e6,e3) + (X,e4,e1) + 8/5*(X,e5,e2)]
5 --> []
6 --> [(X,e6,e1)]
7 --> []
8 --> []
9 --> []
10 --> []
11 --> []
12 --> []
13 --> []
14 --> []


In [152]:
C.set_coker(2)

In [153]:
print(C.coboundary_preim.keys()==C.coboundary_im.keys())
for deg in C.coboundary_preim:
    for wght in C.coboundary_preim[deg]:
        test1=[A.coboundary() for A in C.coboundary_preim[deg][wght]]
        test2=C.coboundary_im[deg][wght]
        if test1!=test2:
            print('Failure at deg =',deg,'wght =',wght,'\n')
            print(test1,'\n')
            print(test2)

True


In [131]:
C.coker[2][0]

[-5/22*(e3,e4,N) + 45/88*(e1,e6,N) + -9/88*(X,e5,e6) + -5/176*(X,e4,e5) + 25/88*(e2,e5,N) + 9/88*(X,e1,e2) + 5/176*(X,e2,e3),
 -6/11*(e3,e4,N) + 5/22*(e1,e6,N) + -1/22*(X,e5,e6) + -3/44*(X,e4,e5) + -7/22*(e2,e5,N) + 1/22*(X,e1,e2) + 3/44*(X,e2,e3),
 11/15*(e1,e2,e3) + -4/15*(e1,e5,e6) + -4/15*(e1,e3,e4) + -4/15*(e1,e4,e5),
 -3/10*(e1,e2,e3) + -3/10*(e1,e5,e6) + 7/10*(e1,e3,e4) + -3/10*(e1,e4,e5),
 -4/15*(e1,e2,e3) + -4/15*(e1,e5,e6) + -4/15*(e1,e3,e4) + 11/15*(e1,e4,e5),
 (e2,e3,e5),
 (e2,e4,e6)]

In [ ]:
# T3=T_symb(3)
# T5=T_symb(5)
# T7=T_symb(7)
# T9=T_symb(9)

# C3=T3.cochain_complex
# C5=T5.cochain_complex
# C7=T7.cochain_complex
# C9=T9.cochain_complex

In [ ]:
# C7.set_harm_basis(2)

In [ ]:
# C7.set_coker(2)

In [ ]:
# for A in C7.neg_harm_basis[2]:
#     print(A,':',len(C7.neg_harm_basis[2][A]))


In [ ]:
#C7.set_coker(2)
#C9.set_coker(2)

In [ ]:
#C3.coker[2]
#C3.coboundary_im[2]
#C3.harm_basis[2]

In [ ]:
#C5.coker[2]
#C5.coboundary_im[2]
#C5.harm_basis[2]

In [ ]:
#C7.coker[2]
#C7.coboundary_im[2]
#C7.harm_basis[2]

In [ ]:
#C9.coker[2]
#C9.coboundary_im[2]
#C9.harm_basis[2]